In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "6"

import torch

from tts.config.ndaligner.training_module_config import NDAlignerTrainingModuleConfigs
from tts.config.utils.io import load_config
from tts.models.ndaligner import init_nd_aligner_training_module

/home/blue2959/monotonic_tts/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## INIT Models

In [37]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
aligner_training_module_cfg_path = "/home/blue2959/monotonic_tts/runs/nd_aligner_main_vctk+libritts+full+sr16k+hop10ms+win25ms_20260807-090119/model_config.json"
aligner_training_module_ckpt_path = "/home/blue2959/monotonic_tts/runs/nd_aligner_main_vctk+libritts+full+sr16k+hop10ms+win25ms_20260807-090119/checkpoints_timit_bae/best_step_timit_bae_0.020014_step_1861000_epoch_100.pth"


model_config = load_config(aligner_training_module_cfg_path, NDAlignerTrainingModuleConfigs,)

aligner_training_module = init_nd_aligner_training_module(
    config=model_config,
    device=device,
)
aligner_training_module.load_checkpoint(
    ckpt_path=aligner_training_module_ckpt_path,
    device=device,
)
aligner = aligner_training_module.nd_aligner.eval()

Loading nested state_dict from key 'model' in /home/blue2959/monotonic_tts/runs/nd_aligner_main_vctk+libritts+full+sr16k+hop10ms+win25ms_20260807-090119/checkpoints_timit_bae/best_step_timit_bae_0.020014_step_1861000_epoch_100.pth
✅ All weights matched perfectly.
Checkpoint loading process finished.


## INIT BenchMarkers (TIMIT)

In [16]:
from silero_vad import load_silero_vad

assert aligner.input_maker is not None

aligner.input_maker.silero_model = load_silero_vad(onnx=True)
aligner.input_maker.trim_nonspeech_region = aligner.input_maker.zero_nonspeech_region = True

In [23]:
# aligner.set_decoding_strategy("viterbi")
aligner.set_decoding_strategy("posterior_viterbi")
# aligner.set_decoding_strategy("mea")

In [24]:
from tts.benchmark.timit.benchmarker import TIMITBenchMarker

TIMIT_ROOT = "/shared/data_zfs/blue2959/TIMIT/TEST"
assert aligner.input_maker is not None

timit_benchmarker = TIMITBenchMarker(
    root_dir=TIMIT_ROOT,
    ref_audio_sr=16_000,
    hyp_audio_sr=model_config.nd_aligner.audio.sr,
    hyp_hop_length=model_config.nd_aligner.audio.hop_length,
    input_maker=aligner.input_maker,
    hyp_ignore_symbols=aligner.input_maker.tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
)

[TIMITBenchMarker] Found 1680 valid (WRD, WAV, TXT) triplets.


In [25]:
with torch.no_grad():
    metrics = timit_benchmarker(
        aligner=aligner,
        max_test_samples=None,
    )

print(f"{metrics.word_boundary_error * 1000:.2f} ms")
print(f"{metrics.p_word_100ms:.2f} %")
print(f"{metrics.p_word_50ms:.2f} %")
print(f"{metrics.p_word_25ms:.2f} %")
print(f"{metrics.p_word_10ms:.2f} %")

Computing Alignments:   0%|          | 0/1680 [00:00<?, ?it/s]

Computing Alignments: 100%|██████████| 1680/1680 [00:50<00:00, 33.02it/s]

19.14 ms
98.70 %
92.36 %
75.26 %
43.99 %


## INIT BenchMarkers (Buckeye)

In [42]:
# aligner.set_decoding_strategy("viterbi")
aligner.set_decoding_strategy("posterior_viterbi")
# aligner.set_decoding_strategy("mea")

In [38]:
# from silero_vad import load_silero_vad

# assert aligner.input_maker is not None

# aligner.input_maker.silero_model = load_silero_vad(onnx=True)
# aligner.input_maker.trim_nonspeech_region = aligner.input_maker.zero_nonspeech_region = True

In [39]:
from tts.benchmark.timit.benchmarker import TIMITBenchMarker
assert aligner.input_maker is not None


BUCKEYE_ROOT = "/shared/data_zfs/blue2959/Buckeye-grid" # (compatible with timit benchmarker!)

buckeye_benchmarker = TIMITBenchMarker(
    root_dir=BUCKEYE_ROOT,
    ref_audio_sr=16_000,
    hyp_audio_sr=model_config.nd_aligner.audio.sr,
    hyp_hop_length=model_config.nd_aligner.audio.hop_length,
    input_maker=aligner.input_maker,
    hyp_ignore_symbols=aligner.input_maker.tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
)

[TIMITBenchMarker] Found 19273 valid (WRD, WAV, TXT) triplets.


In [43]:
with torch.inference_mode():
    metrics = buckeye_benchmarker(
        aligner=aligner,
        max_test_samples=1000,
    )

print(f"{metrics.word_boundary_error * 1000:.2f} ms")
print(f"{metrics.p_word_100ms:.2f} %")
print(f"{metrics.p_word_50ms:.2f} %")
print(f"{metrics.p_word_25ms:.2f} %")
print(f"{metrics.p_word_10ms:.2f} %")

Computing Alignments:   0%|          | 0/1000 [00:00<?, ?it/s]

Computing Alignments: 100%|██████████| 1000/1000 [00:21<00:00, 46.65it/s]

26.39 ms
93.78 %
89.28 %
76.54 %
46.95 %
